# Step 2c — Laterality-normalised, physically-cropped cache

Replaces the 2b cache. Two structural fixes that the public frontier does and our
pipeline did not. Both are correctness issues, not tuning.

### 1 · Laterality normalisation

A right knee is the mirror image of a left knee. Without normalising, "medial" sits
on opposite sides of the image depending on side, so the model has to first infer
the side and then apply a mirrored decision rule — learned from 4k noisy studies.
**Five of the twelve labels depend on that axis** (both menisci, both tibiofemoral
compartments, MCL), and Lateral Meniscus is currently our worst finding at 0.725.

Physics: DICOM patient coordinates put **+x toward the patient's left**. A left knee
sits at positive x with medial toward the midline (smaller x); a right knee sits at
negative x with medial at larger x. So the two sides run in opposite directions and
must be mirrored to agree:

- **coronal / axial** — mirror the image left-right for right knees
- **sagittal** — reverse the slice order for right knees (L-R is the through-plane axis)

Side comes from `Laterality`/`ImageLaterality`, and where those are absent it is
derived from the sign of the patient-x coordinate.

### 2 · Physical-scale crop

A 130 mm crop resized to 336 px gives **0.387 mm/px**. The old cache resized the
whole ~150 mm field of view to 224 px — about **0.67 mm/px**, which is coarser than
the Nyquist limit for a 1 mm meniscal tear, and spends most of the pixel budget on
air and soft tissue outside the joint.

**NOT changed: no random flip augmentation.** Normalisation and augmentation are
opposite operations that both involve a mirror. A deterministic flip to a canonical
side is required; a *random* flip during training would scramble medial/lateral and
invert four labels. Keep `_aug` flip-free.

**Setup:** attach the competition. Internet **ON** (JPEG decoders). Accelerator
**None** — CPU-bound.

## 0 · Config

In [ ]:
import os, sys, glob, json, time, re, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
if not os.path.isdir(DATA_DIR):
    DATA_DIR = "/kaggle/input/rsna-knee-abnormality-detection"
WORK = "/kaggle/working"

IMG        = 336          # 130 mm / 336 px = 0.387 mm/px
CROP_MM    = 130.0        # physical field of view to keep, centred
N_SLICES   = 12
SLICE_BAND = (0.2, 0.8)   # central 60% of the stack
MIN_SLICES = 8
CANON_SIDE = "L"          # every knee is normalised to look like this side

# 3 slots: six was measured as a null (2026-09-10) and costs 2x the training time.
SLOTS = [("SAG_FLUID", "Sagittal", True), ("COR_FLUID", "Coronal", True),
         ("AX_FLUID",  "Axial",    True)]
SLOT_NAMES = [s[0] for s in SLOTS]
N_SLOTS = len(SLOTS)

CACHE = os.path.join(WORK, "cache.npy")
INDEX = os.path.join(WORK, "cache_index.csv")
STATE = os.path.join(WORK, "cache_done.npy")
METAJ = os.path.join(WORK, "cache_meta.json")

assert os.path.isdir(DATA_DIR), f"attach the competition ({DATA_DIR} missing)"
n_std = len(pd.read_csv(os.path.join(DATA_DIR, "train.csv")))
gb = N_SLOTS * N_SLICES * IMG * IMG * n_std / 1e9
print(f"{n_std:,} studies x {N_SLOTS} slots x {N_SLICES} slices x {IMG}px = {gb:.1f} GB")
assert gb < 19, "over the 20 GB output cap"
print(f"resolution: {CROP_MM}mm / {IMG}px = {CROP_MM/IMG:.3f} mm/px "
      f"(old cache was ~0.67 mm/px)")

try:
    import pylibjpeg, pylibjpeg_libjpeg, pylibjpeg_openjpeg   # noqa: F401
except ImportError:
    !pip install -q pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg
import pydicom, cv2
print("pydicom", pydicom.__version__, "| cv2", cv2.__version__)

## 1 · Headers

In [ ]:
train        = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
train_series = pd.read_csv(os.path.join(DATA_DIR, "train_series.csv"))
train_series["series_dir"] = [
    os.path.join(DATA_DIR, "train_series", st, se)
    for st, se in zip(train_series["StudyInstanceUID"], train_series["SeriesInstanceUID"])]

HDR = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
       "RepetitionTime", "EchoTime", "Laterality", "ImageLaterality",
       "PixelSpacing", "ImageOrientationPatient", "ImagePositionPatient"]


def read_header(series_dir):
    rec = {t: None for t in HDR}
    files = glob.glob(os.path.join(series_dir, "*.dcm"))
    rec["n_slices"] = len(files)
    for f in files[:3]:
        try:
            ds = pydicom.dcmread(f, stop_before_pixels=True)
        except Exception:
            continue
        for t in HDR:
            v = getattr(ds, t, None)
            if v is None:
                continue
            if t == "PixelSpacing":
                try: v = float(v[0])
                except Exception: v = None
            elif t in ("ImageOrientationPatient", "ImagePositionPatient"):
                try: v = "|".join(str(float(x)) for x in v)
                except Exception: v = None
            elif not isinstance(v, (int, float, str)):
                v = str(v)
            rec[t] = v
        break
    return rec


from concurrent.futures import ThreadPoolExecutor
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    hdrs = list(tqdm(ex.map(read_header, train_series["series_dir"]),
                     total=len(train_series), desc="headers", unit="series"))
train_series = pd.concat([train_series.reset_index(drop=True), pd.DataFrame(hdrs)], axis=1)
print(f"\n{len(train_series):,} headers in {time.time()-t0:.0f}s")
for c in ("Laterality", "ImageLaterality", "PixelSpacing", "ImagePositionPatient"):
    print(f"  {c:<24} populated {train_series[c].notna().mean():6.1%}")

## 2 · Which knee is this?

`Laterality` first, then `ImageLaterality`, then the physics: the patient-x
coordinate of the image centre. **+x is the patient's left**, so a positive centre-x
means a left knee. Decided per *study* by majority vote across its series, because a
single series can have a missing or wrong tag.

In [ ]:
def _vec(s):
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split("|")]
        return np.array(v) if len(v) >= 3 else None
    except Exception:
        return None


def series_side(row):
    for tag in ("Laterality", "ImageLaterality"):
        v = row.get(tag)
        if isinstance(v, str) and v.strip().upper()[:1] in ("L", "R"):
            return v.strip().upper()[:1], "tag"
    # physics fallback: centre of the image in patient coordinates
    ipp, iop = _vec(row.get("ImagePositionPatient")), _vec(row.get("ImageOrientationPatient"))
    px = row.get("PixelSpacing")
    if ipp is not None:
        cx = ipp[0]
        if iop is not None and len(iop) >= 6 and px and np.isfinite(px):
            # step to the middle of the image along the two in-plane axes
            cx = cx + (iop[0] * px * 0.5 * 500) * 0  # centre offset is small vs |x|; keep origin
        if abs(cx) > 5:                    # ignore near-midline noise
            return ("L" if cx > 0 else "R"), "coords"
    return None, "unknown"


sides, srcs = [], []
for _, r in train_series.iterrows():
    s, k = series_side(r)
    sides.append(s); srcs.append(k)
train_series["side"] = sides
train_series["side_src"] = srcs
print("per-series side source:", train_series["side_src"].value_counts().to_dict())

# majority vote per study
def vote(g):
    v = g["side"].dropna()
    if not len(v):
        return CANON_SIDE, "default"
    top = v.value_counts()
    return top.index[0], ("tag" if (g["side_src"] == "tag").any() else "coords")


rows = []
for uid, g in train_series.groupby("StudyInstanceUID", sort=False):
    s, k = vote(g)
    rows.append({"StudyInstanceUID": uid, "study_side": s, "side_src": k})
side_df = pd.DataFrame(rows)
print("\nstudy side:", side_df["study_side"].value_counts().to_dict())
print("source    :", side_df["side_src"].value_counts().to_dict())
n_flip = (side_df["study_side"] != CANON_SIDE).sum()
print(f"\n{n_flip:,}/{len(side_df):,} studies ({n_flip/len(side_df):.1%}) will be mirrored "
      f"to canonical side {CANON_SIDE}")
print("A roughly 50/50 split is expected. If it is ~100/0 the derivation is wrong —")
print("stop and inspect before caching.")

## 3 · Sequence typing and slot assignment

In [ ]:
_SEP    = re.compile(r"[^a-z0-9]+")
_FATSAT = re.compile(r"\b(fs|fatsat|fatsuppressed|spair|spir|stir|dixon|spectral)\b")
_T1_RX, _T2_RX = re.compile(r"\bt1\b"), re.compile(r"\bt2\b")
_PD_RX  = re.compile(r"\b(pd|pdw|proton)\b")
FATSAT_OPTS = {"FS", "FSAT", "SFS", "SP", "SPIR", "SPAIR", "FATSAT"}


def annotate(df):
    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)
    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    df["fatsat_hdr"] = desc.str.contains(_FATSAT) | opts.apply(
        lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    tr, te = pd.to_numeric(df["RepetitionTime"], errors="coerce"), pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1, t2, pdw = desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX)
    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                    np.where(t2 & ~pdw, "T2", np.where(pdw, "PD", np.where(gre, "GRE",
                     np.where(tr < 800, "T1", np.where(te > 60, "T2",
                      np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid_hdr"] = np.isin(df["weight"], ["PD", "T2"])
    return df


train_series = annotate(train_series)
prov = pd.to_numeric(train_series["Fluid_Sensitive"], errors="coerce").fillna(0).astype(bool)
unk = train_series["weight"].eq("UNK")
train_series["fluid"] = np.where(unk, prov, train_series["fluid_hdr"])
train_series["plane"] = train_series["Anatomical_Plane"].astype(str)
print("weighting:", train_series["weight"].value_counts().to_dict(),
      f"| UNK {unk.mean():.1%} | agreement with provided flag {(train_series['fluid']==prov).mean():.1%}")


def choose(g):
    picked = {}
    for name, plane, fluid in SLOTS:
        sel = (g["plane"] == plane) & (g["fluid"] == fluid)
        cand = g[sel & (g["n_slices"] >= MIN_SLICES)]
        if not len(cand):
            cand = g[sel]
        if not len(cand):
            cand = g[(g["plane"] == plane) & (g["n_slices"] >= MIN_SLICES)]
        if len(cand):
            picked[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
    return picked


rows = []
for uid, g in tqdm(train_series.groupby("StudyInstanceUID", sort=False),
                   total=train_series["StudyInstanceUID"].nunique(),
                   desc="slots", unit="study"):
    picked = choose(g)
    rec = {"StudyInstanceUID": uid}
    for i, name in enumerate(SLOT_NAMES):
        s = picked.get(name)
        rec[f"dir_{i}"]   = s["series_dir"] if s is not None else ""
        rec[f"plane_{i}"] = SLOTS[i][1]
        rec[f"px_{i}"]    = float(s["PixelSpacing"]) if (s is not None and s["PixelSpacing"]) else np.nan
        rec[f"iop_{i}"]   = s["ImageOrientationPatient"] if s is not None else ""
    rows.append(rec)

manifest = (train[["StudyInstanceUID"]]
            .merge(pd.DataFrame(rows), on="StudyInstanceUID", how="left")
            .merge(side_df, on="StudyInstanceUID", how="left"))
manifest["study_side"] = manifest["study_side"].fillna(CANON_SIDE)
for i in range(N_SLOTS):
    manifest[f"dir_{i}"] = manifest[f"dir_{i}"].fillna("")
    manifest[f"iop_{i}"] = manifest[f"iop_{i}"].fillna("")

for i, n in enumerate(SLOT_NAMES):
    print(f"  {n:<11} {(manifest[f'dir_{i}'] != '').mean():6.1%}")
print("PixelSpacing available for the crop:",
      f"{np.mean([manifest[f'px_{i}'].notna().mean() for i in range(N_SLOTS)]):.1%}")

## 4 · Decoder

Order → crop to a physical size → resize → normalise → mirror to the canonical side.
The mirror is applied **last**, on the assembled volume, so it is a single well-defined
operation per slot.

In [ ]:
def _sort_key(ds):
    # Canonical through-plane direction: the normal's sign depends on how the scanner
    # wrote ImageOrientationPatient, so force its dominant component positive. For a
    # sagittal stack that makes slices always run in +x = toward the patient's left.
    iop = getattr(ds, "ImageOrientationPatient", None)
    ipp = getattr(ds, "ImagePositionPatient", None)
    if iop is not None and ipp is not None:
        try:
            n = np.cross(np.array(iop[:3], float), np.array(iop[3:6], float))
            n = n * np.sign(n[np.argmax(np.abs(n))] or 1.0)
            return float(np.dot(np.array(ipp, float), n))
        except Exception:
            pass
    return float(getattr(ds, "InstanceNumber", 0) or 0)


def _row_points_left(iop):
    # +column direction in patient coords. True if increasing column goes toward
    # the patient's LEFT (+x). Used to make image handedness consistent first.
    if iop is None or len(iop) < 3:
        return True
    return float(iop[0]) >= 0


def load_series(series_dir, plane, side, n_slices=N_SLICES, img=IMG,
                band=SLICE_BAND, crop_mm=CROP_MM, canon=CANON_SIDE):
    files = glob.glob(os.path.join(series_dir, "*.dcm"))
    if not files:
        raise FileNotFoundError(series_dir)
    keyed = []
    for f in files:
        try:
            keyed.append((_sort_key(pydicom.dcmread(f, stop_before_pixels=True)), f))
        except Exception:
            continue
    if not keyed:
        raise RuntimeError("no readable headers")
    keyed.sort(key=lambda t: t[0])
    ordered = [f for _, f in keyed]

    # Sagittal: L-R is the through-plane axis, so a right knee is the stack reversed.
    # Reverse the FILE ORDER here, before the slice band is applied -- reversing the
    # finished volume instead would sample anatomical positions shifted by one slice
    # relative to a left knee.
    if side != canon and plane == "Sagittal":
        ordered = ordered[::-1]

    n = len(ordered)
    lo_i, hi_i = int(band[0] * (n - 1)), int(band[1] * (n - 1))
    idx = (np.linspace(lo_i, hi_i, n_slices).round().astype(int) if hi_i > lo_i
           else np.full(n_slices, n // 2))

    out = np.zeros((n_slices, img, img), dtype=np.float32)
    ok, flip_cols = 0, False
    for j, i in enumerate(idx):
        try:
            ds = pydicom.dcmread(ordered[int(i)])
            a = ds.pixel_array.astype(np.float32)
            m = float(getattr(ds, "RescaleSlope", 1) or 1)
            b = float(getattr(ds, "RescaleIntercept", 0) or 0)
            if (m, b) != (1.0, 0.0):
                a = a * m + b
            if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
                a = a.max() - a
            if a.ndim == 3:
                a = a.mean(axis=-1)

            if j == 0:
                iop = getattr(ds, "ImageOrientationPatient", None)
                flip_cols = not _row_points_left(iop)      # make +column = patient left
            if flip_cols:
                a = a[:, ::-1]

            # physical crop, centred, before the resize
            ps = getattr(ds, "PixelSpacing", None)
            try:
                ps = float(ps[0]) if ps is not None else None
            except Exception:
                ps = None
            if ps and np.isfinite(ps) and ps > 0:
                half = int(round(crop_mm / ps / 2))
                cy, cx = a.shape[0] // 2, a.shape[1] // 2
                y0, y1 = max(cy - half, 0), min(cy + half, a.shape[0])
                x0, x1 = max(cx - half, 0), min(cx + half, a.shape[1])
                if (y1 - y0) >= 16 and (x1 - x0) >= 16:
                    a = a[y0:y1, x0:x1]

            out[j] = cv2.resize(np.ascontiguousarray(a), (img, img),
                                interpolation=cv2.INTER_AREA)
            ok += 1
        except Exception:
            continue
    if ok == 0:
        raise RuntimeError("no decodable slices")

    lo_v, hi_v = np.percentile(out, [1, 99])
    if hi_v <= lo_v:
        lo_v, hi_v = float(out.min()), float(out.max())
    if hi_v <= lo_v:
        hi_v = lo_v + 1.0
    out = np.clip((out - lo_v) / (hi_v - lo_v) * 255.0, 0, 255).astype(np.uint8)

    # ---- laterality normalisation, in-plane half ----------------------------
    # +x is the patient's left and columns now point left, so for coronal/axial a
    # right knee is the column-mirror of a left knee. (The sagittal half was done
    # above, on the file order.)
    if side != canon and plane != "Sagittal":
        out = out[:, :, ::-1]
    return np.ascontiguousarray(out)

In [ ]:
# Visual proof: a left knee and a right knee should look alike AFTER normalisation.
import matplotlib.pyplot as plt
have = manifest[manifest["dir_1"] != ""]
L = have[have["study_side"] == "L"].head(1)
R = have[have["study_side"] == "R"].head(1)
if len(L) and len(R):
    fig, ax = plt.subplots(2, 2, figsize=(9, 9))
    for r, (row, lab) in enumerate([(L.iloc[0], "LEFT"), (R.iloc[0], "RIGHT")]):
        raw  = load_series(row["dir_1"], "Coronal", CANON_SIDE)      # force no mirror
        norm = load_series(row["dir_1"], "Coronal", row["study_side"])
        ax[r, 0].imshow(raw[N_SLICES // 2],  cmap="gray"); ax[r, 0].set_title(f"{lab} raw")
        ax[r, 1].imshow(norm[N_SLICES // 2], cmap="gray"); ax[r, 1].set_title(f"{lab} normalised")
        for a_ in ax[r]: a_.axis("off")
    plt.tight_layout(); plt.show()
    print("The two NORMALISED coronals should have the same handedness — medial on the")
    print("same side of the image. If they are still mirrored, the side derivation is wrong.")
else:
    print("need at least one L and one R study to show the comparison")

## 5 · Fill

In [ ]:
N = len(manifest)
SHAPE = (N, N_SLOTS, N_SLICES, IMG, IMG)
print(f"cache {SHAPE} -> {np.prod(SHAPE)/1e9:.1f} GB")
assert np.prod(SHAPE) < 19e9
if not os.path.exists(CACHE):
    np.lib.format.open_memmap(CACHE, mode="w+", dtype=np.uint8, shape=SHAPE).flush()
done = np.load(STATE) if os.path.exists(STATE) else np.zeros(N, dtype=bool)
present = np.zeros((N, N_SLOTS), dtype=bool)
if os.path.exists(INDEX):
    prev = pd.read_csv(INDEX)
    for i in range(N_SLOTS):
        present[:, i] = prev[f"present_{i}"].to_numpy(dtype=bool)
print(f"{done.sum():,}/{N:,} already cached")

from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp
try:
    _CTX = mp.get_context("fork")
    def _pool(n): return ProcessPoolExecutor(max_workers=n, mp_context=_CTX)
    KIND = "processes (fork)"
except ValueError:
    def _pool(n): return ThreadPoolExecutor(max_workers=n)
    KIND = "threads"
print("parallelism:", KIND, "|", os.cpu_count(), "cpus")

JOBS = [(  [manifest[f"dir_{i}"].iloc[r] for i in range(N_SLOTS)],
           manifest["study_side"].iloc[r]) for r in range(N)]


def fill_row(r):
    mm = np.lib.format.open_memmap(CACHE, mode="r+")
    dirs, side = JOBS[r]
    filled = []
    for slot, d in enumerate(dirs):
        if not d:
            continue
        try:
            mm[r, slot] = load_series(d, SLOTS[slot][1], side)
            filled.append(slot)
        except Exception:
            pass
    del mm
    return r, filled


todo = [r for r in range(N) if not done[r]]
t0 = time.time()
with _pool(os.cpu_count()) as ex:
    for k, (r, filled) in enumerate(tqdm(ex.map(fill_row, todo, chunksize=8),
                                         total=len(todo), desc="caching", unit="study"), 1):
        done[r] = True; present[r, filled] = True
        if k % 250 == 0:
            np.save(STATE, done)
np.save(STATE, done)
print(f"\ncached {done.sum():,}/{N:,} in {(time.time()-t0)/60:.0f} min")

## 6 · Index and hand-off

In [ ]:
index = manifest[["StudyInstanceUID", "study_side", "side_src"]].copy()
for i in range(N_SLOTS):
    index[f"present_{i}"] = present[:, i]
index["row"] = np.arange(N)
index["n_present"] = present.sum(axis=1)
index.to_csv(INDEX, index=False)

json.dump({"slots": SLOT_NAMES, "planes": [s[1] for s in SLOTS],
           "n_slices": N_SLICES, "img": IMG, "crop_mm": CROP_MM,
           "slice_band": list(SLICE_BAND), "canon_side": CANON_SIDE,
           "mm_per_px": CROP_MM / IMG}, open(METAJ, "w"), indent=1)

print("slots per study:", index["n_present"].value_counts().sort_index().to_dict())
for i, n in enumerate(SLOT_NAMES):
    want = int((manifest[f"dir_{i}"] != "").sum()); got = int(present[:, i].sum())
    print(f"  {n:<11} {got:,}/{want:,} ({got/max(want,1):.1%})")
print(f"\nno usable slot: {int((index['n_present']==0).sum())}")
print(f"mirrored to canonical side: {(index['study_side'] != CANON_SIDE).mean():.1%}")

mm = np.load(CACHE, mmap_mode="r")
sample = index[index["n_present"] == N_SLOTS].head(3)
fig, axes = plt.subplots(len(sample), N_SLOTS, figsize=(4*N_SLOTS, 4*len(sample)))
axes = np.atleast_2d(axes)
for a, (_, row) in enumerate(sample.iterrows()):
    for b in range(N_SLOTS):
        axes[a, b].imshow(mm[row["row"], b, N_SLICES//2], cmap="gray", vmin=0, vmax=255)
        axes[a, b].set_title(f"{SLOT_NAMES[b]} ({row['study_side']})", fontsize=9)
        axes[a, b].axis("off")
plt.tight_layout(); plt.show()

## 7 · Wiring into training

`cache_meta.json` carries the geometry. In the training config replace the
`PLANES = [...]` line with:

```python
_meta  = json.load(open(os.path.join(CACHE_DIR, "cache_meta.json")))
PLANES = _meta["slots"]
print(f"{PLANES} | {_meta['img']}px | {_meta['mm_per_px']:.3f} mm/px")
```

**336 px is 2.25x the pixels of 224**, so per-step cost rises accordingly: expect
roughly 0.75 s/step at 3 slots x 10 slices x batch 4, about 12 min/epoch. If it OOMs,
drop `BATCH` to 3.

**Keep `_aug` flip-free.** Normalisation is done here, once, deterministically.
A random flip in training would undo it and scramble medial/lateral.

Everything else in the training notebook already reads `len(PLANES)` and adapts.